# Weeks 10–11: Complex Models — When Linear Isn't Enough

## Learning Objectives

By the end of this notebook you will be able to:
- Identify situations where a linear model will fundamentally fail
- Use polynomial features to extend linear models
- Explain how decision trees find non-linear boundaries automatically
- Describe the bias-variance tradeoff with real numbers
- Engineer features that make simpler models more powerful

---

> **The central question:** A linear model draws a straight line (or flat plane).  
> What happens when the world is curved?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.linear_model  import LinearRegression, LogisticRegression
from sklearn.tree          import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline      import Pipeline
from sklearn.metrics       import mean_squared_error, accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
print("Libraries loaded.")

---

## Example 1: Non-linear Relationship — The Inverted-U (Stress vs Performance)

The **Yerkes-Dodson law** says performance peaks at *moderate* stress:  
- Too little stress → bored, not engaged  
- Too much stress → overwhelmed, performance drops  

This is a classic inverted-U (quadratic) relationship that a linear model cannot capture.

In [ ]:
# ── Generate the stress-performance dataset ───────────────────────────────
np.random.seed(0)
x_stress = np.linspace(0, 10, 50)
# True relationship: performance = -a*(stress-5)^2 + peak + noise
y_perf   = -(x_stress - 5)**2 + 25 + np.random.normal(0, 1.5, size=50)

X1 = x_stress.reshape(-1, 1)

# ── Fit models ────────────────────────────────────────────────────────────
# Linear
lin = LinearRegression().fit(X1, y_perf)

# Polynomial degree 2 (manually add x^2 feature)
X1_poly2 = np.column_stack([x_stress, x_stress**2])
poly2 = LinearRegression().fit(X1_poly2, y_perf)

# Polynomial degree 3 (add x^3 feature)
X1_poly3 = np.column_stack([x_stress, x_stress**2, x_stress**3])
poly3 = LinearRegression().fit(X1_poly3, y_perf)

# Predictions on a fine grid for smooth curves
x_grid  = np.linspace(0, 10, 200)
y_lin   = lin.predict(x_grid.reshape(-1,1))
y_p2    = poly2.predict(np.column_stack([x_grid, x_grid**2]))
y_p3    = poly3.predict(np.column_stack([x_grid, x_grid**2, x_grid**3]))

# R² scores
from sklearn.metrics import r2_score
print("=== Model Performance (R²) ===")
print(f"Linear (degree 1):     {r2_score(y_perf, lin.predict(X1)):.3f}")
print(f"Polynomial degree 2:   {r2_score(y_perf, poly2.predict(X1_poly2)):.3f}")
print(f"Polynomial degree 3:   {r2_score(y_perf, poly3.predict(X1_poly3)):.3f}")
print()
print("Degree-2 coefficient on x²:", f"{poly2.coef_[1]:.4f}")
print("→ Negative coefficient on x² confirms the inverted-U shape")

In [ ]:
# ── Manual polynomial features: show WHAT we added ───────────────────────
print("Original feature: x (stress level)")
print("Added feature:    x² (stress squared)")
print()
print(f"{'Stress':>8}  {'x (raw)':>10}  {'x² (added)':>12}")
print("-" * 35)
for s in [0, 2, 5, 7, 10]:
    print(f"{s:8.1f}  {s:10.1f}  {s**2:12.1f}")
print()
print("The x² column is what lets a linear model fit a curve.")
print("The model finds: performance ≈ a·x + b·x² + c")
print("When b is negative, this traces an inverted U.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, y_pred, title, color in zip(
    axes,
    [y_lin, y_p2, y_p3],
    ["Linear (fails)", "Degree 2 (captures it!)", "Degree 3 (slight overfit)"],
    ['crimson', 'steelblue', 'darkorange']
):
    ax.scatter(x_stress, y_perf, alpha=0.6, color='gray', s=25, label='Data')
    ax.plot(x_grid, y_pred, color=color, lw=2.5, label=title)
    ax.axvline(5, color='green', linestyle='--', alpha=0.5, label='Peak stress=5')
    ax.set_xlabel("Stress Level (0-10)")
    ax.set_ylabel("Performance Score")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Example 1: Stress vs Performance — Inverted U", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key insight:**  
Adding a single feature ($x^2$) to the design matrix lets a *linear* model fit a *curved* relationship.  
The model is still "linear in the parameters" — it's linear regression, just with richer features.

---

## Example 2: Threshold Effect — Decision Trees Find Boundaries Naturally

Some real-world relationships flip sharply at a threshold:  
- Below 5 hours studied → fail  
- 5+ hours studied → pass  

A linear boundary (drawn diagonally) will always misclassify boundary cases.  
A decision tree finds the threshold directly.

In [ ]:
# ── Generate study hours + exam score + pass/fail ─────────────────────────
np.random.seed(1)
hours = np.random.uniform(0, 10, 80)
score = 40 + 6*hours + np.random.normal(0, 8, 80)
# Threshold: pass if hours >= 4.5
passed = (hours >= 4.5).astype(int)
# Add some noise (a few students who studied hard still failed, etc.)
noise_idx = np.random.choice(len(hours), size=6, replace=False)
passed[noise_idx] = 1 - passed[noise_idx]

X2 = hours.reshape(-1, 1)

# ── Logistic regression (linear boundary) ─────────────────────────────────
lr_model = LogisticRegression().fit(X2, passed)
lr_acc   = accuracy_score(passed, lr_model.predict(X2))

# ── Decision tree ─────────────────────────────────────────────────────────
dt_model = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X2, passed)
dt_acc   = accuracy_score(passed, dt_model.predict(X2))

print(f"Logistic Regression accuracy: {lr_acc:.3f}")
print(f"Decision Tree accuracy:       {dt_acc:.3f}")
print()
print("Decision Tree Rules:")
print(export_text(dt_model, feature_names=['hours_studied']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

h_grid  = np.linspace(0, 10, 300).reshape(-1, 1)
colors_pt = ['crimson' if p == 0 else 'steelblue' for p in passed]

for ax, model, title in zip(
    axes,
    [lr_model, dt_model],
    ['Logistic Regression (linear boundary)', 'Decision Tree (finds threshold)"']
):
    probs  = model.predict_proba(h_grid)[:, 1]
    ax.scatter(hours, passed + np.random.uniform(-0.05, 0.05, len(hours)),
               c=colors_pt, alpha=0.7, s=40, zorder=3)
    ax.plot(h_grid, probs, lw=2.5, color='black', label='P(pass)')
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Decision threshold')
    ax.axvline(4.5, color='green', linestyle=':', lw=2, alpha=0.8, label='True threshold (4.5h)')
    ax.set_xlabel("Hours Studied")
    ax.set_ylabel("P(Pass)")
    ax.set_title(f"{title}\nAccuracy: {accuracy_score(passed, model.predict(hours.reshape(-1,1))):.3f}",
                 fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    # legend for data points
    ax.scatter([], [], c='crimson',   s=40, label='Fail')
    ax.scatter([], [], c='steelblue', s=40, label='Pass')
    ax.legend(fontsize=8)

plt.suptitle("Example 2: Hours Studied vs Pass/Fail — Threshold Effect", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Example 3: The XOR Problem — Feature Engineering Saves the Day

XOR: class 1 if both inputs are the same sign, class 0 otherwise.

```
Top-right (++):  class 1      Bottom-right (+-): class 0
Bottom-left (--): class 1     Top-left    (-+): class 0
```

No straight line can separate these groups. But the **product feature** $x_1 \cdot x_2$ solves it instantly:  
- $x_1 \cdot x_2 > 0$ → same quadrant → class 1  
- $x_1 \cdot x_2 < 0$ → different quadrants → class 0

In [ ]:
# ── Generate XOR-like data ─────────────────────────────────────────────────
np.random.seed(3)
n = 200
x1 = np.random.uniform(-2, 2, n)
x2 = np.random.uniform(-2, 2, n)
# XOR label: same quadrant → 1, different quadrant → 0
y_xor = ((x1 * x2) > 0).astype(int)
# Add small noise
flip = np.random.rand(n) < 0.05
y_xor[flip] = 1 - y_xor[flip]

X3_raw = np.column_stack([x1, x2])

# ── Linear model on raw features ──────────────────────────────────────────
lr_xor = LogisticRegression().fit(X3_raw, y_xor)
lr_xor_acc = accuracy_score(y_xor, lr_xor.predict(X3_raw))
print(f"Logistic regression on raw [x1, x2]: accuracy = {lr_xor_acc:.3f}")
print("→ Near 50% — essentially random guessing!")

# ── Add interaction feature x1*x2 ─────────────────────────────────────────
x_interact = (x1 * x2).reshape(-1, 1)   # the key interaction term
X3_eng     = np.column_stack([x1, x2, x1*x2])

lr_xor_eng = LogisticRegression().fit(X3_eng, y_xor)
lr_eng_acc = accuracy_score(y_xor, lr_xor_eng.predict(X3_eng))
print(f"\nLogistic regression on [x1, x2, x1*x2]: accuracy = {lr_eng_acc:.3f}")
print("→ Near 100% — one new feature completely solved the problem!")

In [ ]:
# ── Visualize decision boundaries ─────────────────────────────────────────
def plot_boundary(ax, model, X_full, feature_idx, y, title, feature_cols='all'):
    """Plot decision boundary for a 2D slice of feature space."""
    x_min, x_max = X_full[:, 0].min() - 0.3, X_full[:, 0].max() + 0.3
    y_min, y_max = X_full[:, 1].min() - 0.3, X_full[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid_2d = np.c_[xx.ravel(), yy.ravel()]
    if feature_cols == 'with_interact':
        grid_full = np.column_stack([grid_2d, grid_2d[:,0]*grid_2d[:,1]])
    else:
        grid_full = grid_2d
    Z = model.predict(grid_full).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    scatter = ax.scatter(X_full[:, 0], X_full[:, 1], c=y,
                         cmap='RdBu', edgecolors='k', linewidths=0.4, s=30, alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x₁"); ax.set_ylabel("x₂")
    ax.grid(alpha=0.2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_boundary(axes[0], lr_xor,     X3_raw, None, y_xor,
              f"Logistic Regression (raw)\nAccuracy: {lr_xor_acc:.3f}")
plot_boundary(axes[1], lr_xor_eng, X3_raw, None, y_xor,
              f"Logistic Regression + x₁×x₂\nAccuracy: {lr_eng_acc:.3f}",
              feature_cols='with_interact')

plt.suptitle("Example 3: XOR Problem — Interaction Feature Solves It", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: x₁×x₂ is positive when both are same sign (same class in XOR).")
print("One engineered feature turned an unsolvable problem into a trivially solvable one.")

---

## Example 4: House Price vs. Size — Diminishing Returns

Small houses gain a lot of value per square foot.  
Large houses gain much less per additional square foot.  
This **diminishing returns** shape is everywhere in economics.

In [ ]:
# ── Generate house price data with diminishing returns ────────────────────
np.random.seed(7)
sqft  = np.sort(np.random.uniform(300, 3000, 80))
# True relationship: concave (diminishing returns)
price = 50 * sqft**0.65 + np.random.normal(0, 15000, 80)
price = np.clip(price, 50000, None)   # no negative prices

X4 = sqft.reshape(-1, 1)
x4_grid = np.linspace(300, 3000, 300).reshape(-1, 1)

# ── Fit models ────────────────────────────────────────────────────────────
lin4 = LinearRegression().fit(X4, price)

# Build polynomial feature sets
def poly_pipeline(degree):
    return Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr',   LinearRegression())
    ])

poly2_4  = poly_pipeline(2).fit(X4, price)
poly4_4  = poly_pipeline(4).fit(X4, price)
poly10_4 = poly_pipeline(10).fit(X4, price)

print("=== Train R² Scores ===")
for name, model, X_input in [
    ("Linear",       lin4,    X4),
    ("Poly degree 2", poly2_4,  X4),
    ("Poly degree 4", poly4_4,  X4),
    ("Poly degree 10", poly10_4, X4),
]:
    r2 = r2_score(price, model.predict(X_input))
    print(f"  {name:20s}: R² = {r2:.4f}")
print()
print("Note: degree 10 achieves high R² by memorizing noise — watch for overfitting.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: compare all degrees
axes[0].scatter(sqft, price, alpha=0.4, s=25, color='gray', label='Data')
for model, label, color, lw in [
    (lin4,    'Linear',       'crimson',    2),
    (poly2_4, 'Degree 2',     'steelblue',  2),
    (poly4_4, 'Degree 4',     'darkorange', 2),
    (poly10_4,'Degree 10',    'purple',     2),
]:
    axes[0].plot(x4_grid, model.predict(x4_grid), label=label, color=color, lw=lw)
axes[0].set_xlabel("House Size (sqft)")
axes[0].set_ylabel("Price ($)")
axes[0].set_title("Polynomial Degrees — How Model Complexity Grows", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Panel 2: zoom in on degree-10 overfitting
axes[1].scatter(sqft, price, alpha=0.4, s=25, color='gray', label='Data')
axes[1].plot(x4_grid, poly2_4.predict(x4_grid),  label='Degree 2 (good fit)', color='steelblue', lw=2)
axes[1].plot(x4_grid, poly10_4.predict(x4_grid), label='Degree 10 (overfit!)', color='purple',   lw=2)
axes[1].set_xlabel("House Size (sqft)")
axes[1].set_ylabel("Price ($)")
axes[1].set_title("Degree 2 vs Degree 10 — Overfitting", fontsize=11)
axes[1].set_ylim(-50000, 700000)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle("Example 4: House Price vs Size — Diminishing Returns", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 2: The Bias-Variance Tradeoff

This is one of the most important concepts in all of machine learning.

| | **High Bias (Underfitting)** | **Low Bias / High Variance (Overfitting)** |
|---|---|---|
| **Train error** | High | Very low |
| **Test error** | High | High |
| **Model** | Too simple | Too complex |
| **Fix** | More features, higher polynomial degree | Regularization, more data, simpler model |

**The sweet spot** has low bias AND low variance — but you can't minimize both simultaneously.

In [ ]:
# ── Generate data and split train/test ────────────────────────────────────
np.random.seed(10)
n_total = 100
x_bv = np.sort(np.random.uniform(0, 10, n_total))
y_bv = np.sin(x_bv) + 0.3*x_bv + np.random.normal(0, 0.5, n_total)

X_bv = x_bv.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X_bv, y_bv, test_size=0.3, random_state=0)
print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

# ── Fit polynomial models of increasing degree ────────────────────────────
degrees = [1, 2, 3, 5, 9]
train_errors, test_errors = [], []

print(f"\n{'Degree':>8}  {'Train RMSE':>12}  {'Test RMSE':>12}  {'Status':>15}")
print("-" * 55)

for deg in degrees:
    model = poly_pipeline(deg).fit(X_train, y_train)
    tr_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    te_rmse = np.sqrt(mean_squared_error(y_test,  model.predict(X_test)))
    train_errors.append(tr_rmse)
    test_errors.append(te_rmse)

    if deg <= 2:
        status = 'Underfitting'
    elif deg == 3:
        status = '← Sweet spot'
    else:
        status = 'Overfitting'
    print(f"{deg:>8d}  {tr_rmse:12.4f}  {te_rmse:12.4f}  {status:>15}")

In [ ]:
# ── Decision tree depth bias-variance ─────────────────────────────────────
depths = [1, 2, 3, 5, 10]
dt_train_errors, dt_test_errors = [], []

print("=== Decision Tree Depth ===")
print(f"{'Depth':>8}  {'Train RMSE':>12}  {'Test RMSE':>12}  {'Status':>15}")
print("-" * 55)

for depth in depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=0)
    dt.fit(X_train, y_train)
    tr_rmse = np.sqrt(mean_squared_error(y_train, dt.predict(X_train)))
    te_rmse = np.sqrt(mean_squared_error(y_test,  dt.predict(X_test)))
    dt_train_errors.append(tr_rmse)
    dt_test_errors.append(te_rmse)

    if depth <= 2:
        status = 'Underfitting'
    elif depth == 3:
        status = '← Sweet spot'
    else:
        status = 'Overfitting'
    print(f"{depth:>8d}  {tr_rmse:12.4f}  {te_rmse:12.4f}  {status:>15}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: polynomial bias-variance
axes[0].plot(degrees, train_errors, 'b-o', lw=2, ms=7, label='Train RMSE')
axes[0].plot(degrees, test_errors,  'r-s', lw=2, ms=7, label='Test RMSE')
axes[0].axvspan(0.5, 2.5, alpha=0.12, color='orange', label='Underfitting')
axes[0].axvspan(2.5, 3.5, alpha=0.12, color='green',  label='Sweet spot')
axes[0].axvspan(3.5, 9.5, alpha=0.12, color='red',    label='Overfitting')
axes[0].set_xticks(degrees)
axes[0].set_xlabel("Polynomial Degree")
axes[0].set_ylabel("RMSE")
axes[0].set_title("Polynomial — Bias-Variance Tradeoff", fontsize=12)
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
axes[0].annotate('Underfitting\n(high bias)', xy=(1, train_errors[0]+0.05),
                 fontsize=9, color='darkorange', ha='center')
axes[0].annotate('Overfitting\n(high variance)', xy=(7, test_errors[-1]+0.05),
                 fontsize=9, color='red', ha='center')

# Panel 2: decision tree bias-variance
axes[1].plot(depths, dt_train_errors, 'b-o', lw=2, ms=7, label='Train RMSE')
axes[1].plot(depths, dt_test_errors,  'r-s', lw=2, ms=7, label='Test RMSE')
axes[1].axvspan(0.5, 2.5, alpha=0.12, color='orange')
axes[1].axvspan(2.5, 3.5, alpha=0.12, color='green')
axes[1].axvspan(3.5, 10.5, alpha=0.12, color='red')
axes[1].set_xticks(depths)
axes[1].set_xlabel("Tree Depth")
axes[1].set_ylabel("RMSE")
axes[1].set_title("Decision Tree — Bias-Variance Tradeoff", fontsize=12)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle("Bias-Variance Tradeoff: Train vs Test Error", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**The bias-variance tradeoff in one sentence:**  
Train error always decreases with model complexity. Test error forms a U-shape — the minimum of that U is your target.

---

## Part 3: Decision Tree Intuition — Visual Splitting

A decision tree works by asking binary questions that split the data.  
At each step it finds the split that most cleanly separates the classes.

In [ ]:
# ── 2D classification problem to visualize splits ─────────────────────────
np.random.seed(5)
n_pts = 120
size_sqft  = np.random.uniform(500, 3000, n_pts)
num_rooms  = np.random.uniform(1, 8, n_pts)
# High price if large OR many rooms
price_class = ((size_sqft > 1200) & (num_rooms > 2.5)).astype(int)
# Add some noise
flip2 = np.random.rand(n_pts) < 0.05
price_class[flip2] = 1 - price_class[flip2]

X_dt2 = np.column_stack([size_sqft, num_rooms])

# Fit trees with increasing depth
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, depth in zip(axes, [1, 2, 5]):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_dt2, price_class)
    acc = accuracy_score(price_class, dt.predict(X_dt2))

    # Create decision boundary
    xx, yy = np.meshgrid(np.linspace(500, 3000, 200), np.linspace(1, 8, 200))
    Z = dt.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(size_sqft, num_rooms, c=price_class, cmap='RdBu',
               edgecolors='k', linewidths=0.4, s=30, zorder=3)
    ax.set_xlabel("Size (sqft)")
    ax.set_ylabel("Number of Rooms")
    ax.set_title(f"Decision Tree — Depth {depth}\nAccuracy: {acc:.3f}", fontsize=11)
    ax.grid(alpha=0.2)

plt.suptitle("Decision Tree Splits: Depth 1 vs 2 vs 5", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Print human-readable decision rules ───────────────────────────────────
dt_readable = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_dt2, price_class)
print("=== Decision Tree Rules (depth 3) ===")
print(export_text(dt_readable, feature_names=['size_sqft', 'num_rooms']))
print()
print("These rules are immediately interpretable:")
print("  IF size > 1200 AND rooms > 2.5 THEN price_class = high")
print("This is why decision trees are popular in regulated industries (banking, medicine).")

---

## Part 4: Feature Engineering — Smarter Features Beat Complex Models

Sometimes adding **domain-knowledge-based** features beats adding polynomial terms blindly.

In [ ]:
# ── House price regression with feature engineering ───────────────────────
np.random.seed(20)
n = 100
area_sqft = np.random.uniform(400, 3000, n)
num_rooms = np.random.randint(1, 8, n).astype(float)

# True price depends on area per room (efficiency) and whether it's large
area_per_room = area_sqft / num_rooms
is_large      = (area_sqft > 1500).astype(float)
y_house = 200 * area_per_room + 30000 * is_large + 50000 + np.random.normal(0, 8000, n)

X_house_split = train_test_split(
    np.column_stack([area_sqft, num_rooms, area_per_room, is_large]),
    y_house, test_size=0.3, random_state=0
)
X_tr, X_te, y_tr, y_te = X_house_split

# Model 1: Raw features only
m1 = LinearRegression().fit(X_tr[:, :2], y_tr)

# Model 2: Raw + engineered features
m2 = LinearRegression().fit(X_tr, y_tr)

# Model 3: Polynomial (degree 2) on raw features
m3 = poly_pipeline(2).fit(X_tr[:, :2], y_tr)

print("=== Test Set RMSE Comparison ===")
results = {
    "Raw features [area, rooms]": m1.predict(X_te[:, :2]),
    "+ Engineered features":      m2.predict(X_te),
    "Polynomial deg-2 (raw)": m3.predict(X_te[:, :2]),
}
for name, preds in results.items():
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2   = r2_score(y_te, preds)
    print(f"  {name:38s}: RMSE = ${rmse:8.0f},  R² = {r2:.4f}")

print()
print("→ Engineered features (area_per_room, is_large) beat polynomial features")
print("  because they encode domain knowledge about how price is actually determined.")

In [ ]:
# ── Visualize feature importance ──────────────────────────────────────────
all_features = ['area_sqft', 'num_rooms', 'area_per_room', 'is_large']
coeffs = m2.coef_
# Normalize for comparison (multiply by feature std to get standardized coeff)
feature_stds = X_tr.std(axis=0)
standardized = np.abs(coeffs * feature_stds)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart of standardized coefficients
colors = ['steelblue', 'steelblue', 'darkorange', 'darkorange']
bars = axes[0].bar(all_features, standardized, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title("Feature Importance (Standardized Coefficients)", fontsize=11)
axes[0].set_ylabel("|Coefficient × Std|")
axes[0].grid(alpha=0.3, axis='y')
for bar, val in zip(bars, standardized):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f"{val:.0f}", ha='center', fontsize=9)
# Add legend
raw_patch = mpatches.Patch(color='steelblue',   label='Raw features')
eng_patch = mpatches.Patch(color='darkorange',  label='Engineered features')
axes[0].legend(handles=[raw_patch, eng_patch], fontsize=9)

# Predicted vs actual
y_pred_m2 = m2.predict(X_te)
axes[1].scatter(y_te, y_pred_m2, alpha=0.6, color='mediumseagreen', s=40)
min_v, max_v = min(y_te.min(), y_pred_m2.min()), max(y_te.max(), y_pred_m2.max())
axes[1].plot([min_v, max_v], [min_v, max_v], 'r--', lw=1.5, label='Perfect prediction')
axes[1].set_xlabel("Actual Price")
axes[1].set_ylabel("Predicted Price")
axes[1].set_title("Model with Engineered Features\nPredicted vs Actual", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle("Feature Engineering: Domain Knowledge > Polynomial Complexity", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Summary

| Concept | Key Point |
|---------|----------|
| **When linear fails** | Non-linear relationships, threshold effects, XOR-like interactions |
| **Polynomial features** | Add $x^2, x^3, \ldots$ to design matrix — still linear regression internally |
| **Feature engineering** | Hand-crafted features from domain knowledge often outperform brute-force polynomial expansion |
| **Decision trees** | Find thresholds automatically, naturally handle non-linearity, easy to interpret |
| **Bias-variance tradeoff** | Train error ↓ with complexity, test error has a U-shape — aim for the minimum |
| **Underfitting** | Model too simple: high train error AND high test error |
| **Overfitting** | Model too complex: low train error but high test error |
| **Sweet spot** | Model complexity where test error is minimized |

---

**Next Week:** We'll see how stacking non-linear layers (neural networks) solves all of these problems at once — at the cost of interpretability.